# Calibration X6Y3 — the FULL flow on the real chip (ZCU216)

The complete X6Y3 calibration chain (spec [two-qubit/04](../specs/two-qubit/04-x6y3-fixed-frequency.md)
§2) on a **real ZCU216** driving the fixed-frequency 8-qubit ring — readout → 1Q GE → EF subspace →
two-qubit (JAZZ + the two-qubit-drive CZ chain per ring pair) → an R-ladder validation. It extends
[`calibration_x6y3.ipynb`](calibration_x6y3.ipynb) (the readout + 1Q chain, spec 13) with the EF and
two-qubit sections X0–X4 landed; the config of record is the real X6Y3 qcal tree
([`cal-config-x6y3.yaml`](cal-config-x6y3.yaml)) — 8 qubits, ring pairs (0, 1) … (6, 7), (7, 0), the
**two-qubit-drive** CZ form (both tones on the pair's own gate channels at `CZ/freq`, spec 04 §1),
(5, 6)/(6, 7) EF-shelved sandwich pairs — loaded with `Config.from_qcal`, written back with
`save_qcal` every cell.

Like the other hardware notebooks this connects over `RemoteDriver` and is **not executed in CI** —
its co-sim twin ([`calibration_x6y3_full_cosim.ipynb`](calibration_x6y3_full_cosim.ipynb)) runs the
same chain end-to-end against planted models and is the verified reference for every code path here.

| flow stage (spec 04 §2) | here |
|---|---|
| readout tune (walkthrough 1.3–1.5) | `ReadoutCalibration` → `Separation` → `Fidelity` → `ReadoutFidelity`, all 8, heralded (the spec-13 chain at the reference knobs) |
| 1Q GE (2, 3.1–3.2) | `Frequency` (V-fit) → `Amplitude` coarse + fine → `Phase` relative loop + absolute pass |
| EF subspace (3.3) | 3-level `ClassifierN` training (|0⟩/|1⟩/|2⟩ RAW clouds) → `EFFrequency` → `EFAmplitude` (X90) → `EFAmplitude(gate='X')` → `EFPhase` |
| ZZ (4.1) | `JAZZ` around the ring — on fixed-frequency it *characterizes* the always-on ZZ (no null knob) |
| CZ tune-up (4.2–4.4) | per pair: `CZSweep('freq')` → `CZFrequency` → `RelativePhase` → `CZAmplitude` (short ladder) → `LocalPhases`; then `SpectatorPhase` for every ring neighbour the pair's pulse list carries |
| validation (4.5) | conditionality R at an amplified gate count per pair |
| N/A on this chip | everything flux (arc/park, parametric coupler) — X6Y3 has no tunable element |

**Run order matters** (spec 04 §5 / X4): the EF section calibrates every qubit's `EF/freq` +
`EF/x/amp` — q6's EF X is the **prerequisite** of the (5, 6)/(6, 7) sandwich CZ chain (the shelving
pre/post-pulse `single_qubit/6/EF/X/pulse`), so the EF section must complete before those two pairs.
The CZ chain runs plain pairs first, sandwich pairs after.

In [ ]:
import shutil
from pathlib import Path

import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt

import riscq
from riscq import run as rq
from riscq.cal import (Amplitude, CZAmplitude, CZFrequency, CZSweep, ClassifierN, Config,
                       EFAmplitude, EFFrequency, EFPhase, Fidelity, Frequency, JAZZ, LocalPhases,
                       Phase, ReadoutCalibration, ReadoutFidelity, RelativePhase, Separation,
                       SpectatorPhase, calc_cz_frequency, kernels, pair_key)
from riscq.cal.base import (GATE_CH, SEP, acquire_shots, batch_timeout, ef_table, grid_period,
                            readout_tables, relax_batches, x90_vz)
from riscq.cal.readout import _rawiq_prog
from riscq.cal.twoqubit import _cz_cond_R, _cz_freq_word
from riscq.driver.remote import RemoteDriver, upload_bundle
from riscq.lang import Array, compile_kernel
from riscq.map import LEAD, SocMap, SocParams
from riscq.pulses import units

MHz, GHz, us = 1e6, 1e9, 1e-6

## Connect to the board and the config of record

The chassis is the ZCU216 board server ([docs/software/board-server.md](../docs/software/board-server.md))
with an X6Y3 gateware bundle loaded; the config is the real qcal tree, copied to a **working file**
that every cell reloads (the reference's `cfg.load()`) and every applied proposal is saved back into.

In [ ]:
BOARD = '192.168.1.122'                   # the ZCU216's LAN address (or the full PYRO: uri)
PORT = 9091

SW = Path(riscq.__file__).resolve().parents[1]              # .../software
SRC = SW.parent / 'examples' / 'cal-config-x6y3.yaml'       # the real X6Y3 qcal tree
WORK = SW / 'build' / 'x6y3_full_config.yaml'               # the working copy
WORK.parent.mkdir(exist_ok=True)
shutil.copy(SRC, WORK)

drv = RemoteDriver(BOARD, PORT)
print('server:', drv.board.info())

# first time only — push the X6Y3 build up and load it:
# upload_bundle(drv, 'x6y3', xsa='../build/x6y3/top.xsa',
#               params_json='../software/configs/x6y3.json')
# info = drv.board.load('x6y3')
# assert info['mts_result'] == 0, 'multi-tile sync missed its target latencies'

m = SocMap(SocParams.from_json(drv.board.get_params()))
QUBITS = list(range(8))
PAIRS = [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 0)]   # the ring
SANDWICH = [(5, 6), (6, 7)]                    # EF-shelved pairs (q6 is the shelf, spec 04 §1)
PLAIN = [p for p in PAIRS if p not in SANDWICH]

cfg = Config.from_qcal(WORK)
cfg.check_hardware(m.params)     # the tree's DAC/ADC rates + interpolation must describe THIS bundle
print(f"connected to '{m.params.name}': {m.params.qubit_num} cores;  herald = {cfg['readout/herald']}")

def step(cal):
    '''run -> print -> apply if ok -> persist to the working tree (qcal's auto write-back).'''
    r = cal.run(drv)
    print(f'{r.label}: ok={r.ok}  proposal={r.proposal}')
    if r.ok:
        r.apply()
        r.cfg.save_qcal(WORK)
    else:
        print('  fit failed — config left unchanged (fail-loud, spec 13 §2)')
    return r

# Readout

`ReadoutCalibration` → `Separation` → `Fidelity` → `ReadoutFidelity` — exactly
[`calibration_x6y3.ipynb`](calibration_x6y3.ipynb)'s spec-13 chain and knobs (see it for the
step-by-step commentary); all 8 qubits simultaneously on the frequency-multiplexed readout,
heralded. The trained discriminator (each qubit's `demod/phase` + `res_sign`) is what every
counts-mode step below reads.

In [ ]:
cfg = Config.from_qcal(WORK)
rc = step(ReadoutCalibration(cfg, QUBITS, shots=1000, gate='X90'))

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for q, ax in zip(QUBITS, axes.flat):
    d = rc.data[q]
    ax.scatter(d['iq0'][:, 0], d['iq0'][:, 1], s=4, label='|0>')
    ax.scatter(d['iq1'][:, 0], d['iq1'][:, 1], s=4, label='|1>')
    ax.set_title(f"q{q}  sep={d['separation']:.2f}")
    ax.set_aspect('equal'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
cfg = Config.from_qcal(WORK)
sep = step(Separation(cfg, QUBITS, span=2.5 * MHz, points=31, shots=32, gate='X90'))

for q in QUBITS:
    plt.plot((sep.data[q]['x'] - sep.data[q]['x'].mean()) / MHz, sep.data[q]['y'], '-', label=f'q{q}')
plt.xlabel('readout freq − centre [MHz]'); plt.ylabel('cluster SNR'); plt.legend(ncol=4); plt.show()

In [ ]:
cfg = Config.from_qcal(WORK)
fid = step(Fidelity(cfg, QUBITS, amp_span=0.005, points=31, shots=3000, gate='X90'))

for q in QUBITS:
    plt.plot(fid.data[q]['x'], fid.data[q]['y'], '-', label=f'q{q}')
plt.xlabel('readout amp'); plt.ylabel('confusion diagonal'); plt.legend(ncol=4); plt.show()

In [ ]:
cfg = Config.from_qcal(WORK)
rof = step(ReadoutFidelity(cfg, QUBITS, shots=5000, gate='X90'))

for q in QUBITS:
    print(f"q{q} confusion (row = prepared, col = classified):")
    print(np.round(rof.data[q]['confusion'], 3), f"  fidelity={rof.data[q]['fidelity']:.3f}")

# Single qubit — GE

`Frequency` (Ramsey V-fit, the reference's ±2.5/±5 MHz detunings), `Amplitude` coarse + relative
fine, `Phase` (per-qubit relative loop, then the wider absolute pass) — spec 13's chain, here on
**all 8 qubits** (the full flow calibrates the whole ring; the reference notebook demoed subsets).

In [ ]:
cfg = Config.from_qcal(WORK)
detunings = np.array([-5, -2.5, 2.5, 5]) * MHz          # the reference's exact set
fr = step(Frequency(cfg, QUBITS, detunings=detunings, t_max=1 * us, points=30, shots=512))

for q in QUBITS:
    d = fr.data[q]
    plt.plot(d['applied'], d['obs'], 'o', label=f'q{q}')
plt.xlabel('applied detuning [codes]'); plt.ylabel('|fringe| [codes]'); plt.legend(ncol=4); plt.show()
for q in QUBITS:
    print(f"qubit/{q}/freq -> {cfg[f'qubit/{q}/freq'] / GHz:.6f} GHz")

In [ ]:
cfg = Config.from_qcal(WORK)
ac = step(Amplitude(cfg, QUBITS, gate='X90', n_gates=1, amp_span=(0.03, 0.97), points=31, shots=512))

cfg = Config.from_qcal(WORK)
af = step(Amplitude(cfg, QUBITS, gate='X90', n_gates=4, amp_span=(0.7, 1.3), relative_amp=True,
                    points=31, shots=512))
for q in QUBITS:
    print(f"q{q}: X90 amp -> {cfg[f'qubit/{q}/x90/amp']:.5f}")

In [ ]:
cfg = Config.from_qcal(WORK)
for q in QUBITS:                                 # the reference's per-qubit relative loop
    step(Phase(cfg, [q], span=0.25, points=21, shots=512, relative_phase=True))

cfg = Config.from_qcal(WORK)
ph = step(Phase(cfg, [1, 3, 5, 7], span=0.3, points=31, shots=512))   # the wider absolute pass

for q in QUBITS:
    vz = cfg.get(f'qubit/{q}/x90/vz', [0.0, 0.0])
    print(f'q{q}: X90 virtual-Z pair -> [{vz[0]:+.4f}, {vz[1]:+.4f}] rad')

# EF subspace

The |1⟩→|2⟩ manifold (walkthrough 3.3, spec 04 §2 / X4). The hardware `res` bit is 2-level, so the
EF calibrations read P(|2⟩) host-side through a pre-trained 3-level **`ClassifierN`** — trained here
from three RAW reference clouds per qubit: |0⟩ (no prep) and |1⟩ (X90·X90) through the 2-level RAW
program, |2⟩ through a GE π followed by the tree's **stored EF X** (`k_ef_rabi` with the amp sweep
pinned — X6Y3's config carries a calibrated EF X, so the |2⟩ prep is already good enough to seed a
cluster; the section then refines every EF knob). All captures run in the demod **zero-phase**
frame, the same frame `ReadoutCalibration` trains in (spec 13 §5).

RAW shots are core-RAM-bounded (spec 13 §5): every EF class below keeps `points·shots ≈ 1000`
(≈ 8 KB of the 16 KB out-buffer budget).

In [ ]:
cfg = Config.from_qcal(WORK)

def train_classifier3(qubits, shots=400):
    '''{q: ClassifierN} from |0>/|1>/|2> RAW reference clouds (all qubits in parallel per level).'''
    clouds = {q: [] for q in qubits}
    progs, timeout = {}, 0
    for q in qubits:                              # |0> and |1>: the 2-level RAW program (prep scalar)
        prog, period = _rawiq_prog(m, cfg, q, 'X90', shots)
        progs[q] = prog
        timeout = max(timeout, batch_timeout(shots * period))
    rq.setup(drv, m, progs)
    for level in (0, 1):
        iq = acquire_shots(drv, m, progs, level, shots, timeout)
        for q in qubits:
            clouds[q].append(iq[q])
    progs, timeout = {}, 0
    for q in qubits:                              # |2>: GE pi then the STORED EF X, captured RAW
        table, ge_freq, ef_freq = ef_table(cfg, q, m, 'x')
        ro, demod, code, dur, ddly = readout_tables(cfg, q, m, phase=0.0)   # the zero-phase frame
        ge = table.pulses['x90'].dur_batches(m, GATE_CH)
        ef = table.pulses['ef'].dur_batches(m, GATE_CH)
        period = grid_period(relax_batches(cfg, m), SEP + ef + LEAD + 2 * ge, dur, ddly)
        progs[q] = compile_kernel(kernels.k_ef_rabi, m, tables=dict(gate=table, ro=ro, demod=demod),
                                  out=Array(2 * shots), npts=1, shots=shots, period=period,
                                  ngates=1, code=code, ddly=ddly, ge_freq=ge_freq, ef_freq=ef_freq,
                                  **x90_vz(cfg, q))
        timeout = max(timeout, batch_timeout(shots * period))
    par = {q: {'a0q': units._amp_code(float(cfg[f'qubit/{q}/EF/x/amp'])) << 16, 'daq': 0}
           for q in qubits}
    out = rq.run(drv, m, progs, params=par, results=['out'], timeout=timeout)
    for q in qubits:
        clouds[q].append(out[q]['out'].reshape(shots, 2).astype(float))
    return {q: ClassifierN(clouds[q]) for q in qubits}

CLF = train_classifier3(QUBITS)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for q, ax in zip(QUBITS, axes.flat):
    for lvl, c in enumerate(CLF[q].clusters):
        ax.scatter(c[:, 0], c[:, 1], s=4, label=f'|{lvl}>')
    ax.set_title(f'q{q}  sep={CLF[q].separation:.2f}')
    ax.set_aspect('equal'); ax.legend()
plt.tight_layout(); plt.show()

## EF Frequency

The EF Ramsey V-fit (`Frequency(subspace='EF')` parity): GE π prep, retune to `EF/freq`, two EF X90s
around a swept wait with a virtual-Z detuning, P(|2⟩) fringes → `a·|x − b| + c`. Writes
`qubit/{q}/EF/freq`.

In [ ]:
cfg = Config.from_qcal(WORK)
efr = step(EFFrequency(cfg, QUBITS, CLF, detune=5 * MHz, n_detune=4, points=14, shots=48))

for q in QUBITS:
    print(f"qubit/{q}/EF/freq -> {cfg[f'qubit/{q}/EF/freq'] / GHz:.6f} GHz")

## EF Amplitude — X90, then X

The EF Rabi (`Amplitude(subspace='EF')` parity): GE π prep, swept EF drive, P(|2⟩) cosine →
`EF/x90/amp` + the EF Rabi rate; the second pass calibrates the EF **π** (`gate='X'`) —
`EF/x/amp`, **the pulse the (5, 6)/(6, 7) sandwich shelves with** (its prerequisite).

In [ ]:
cfg = Config.from_qcal(WORK)
efa = step(EFAmplitude(cfg, QUBITS, CLF, gate='X90', n_gates=1, points=21, shots=48))

cfg = Config.from_qcal(WORK)
efx = step(EFAmplitude(cfg, QUBITS, CLF, gate='X', n_gates=1, points=21, shots=48))
for q in QUBITS:
    print(f"q{q}: EF x90 amp -> {cfg[f'qubit/{q}/EF/x90/amp']:.5f}   "
          f"EF x amp -> {cfg[f'qubit/{q}/EF/x/amp']:.5f}")

## EF Phase

The EF X90's virtual-Z pair (`Phase(subspace='EF')` parity, X4): the two Rz(±π/2)-decorated
three-EF-X90 sequences on a GE-π-prepped qubit, P(|2⟩) line crossing → `qubit/{q}/EF/x90/vz`
(one crossing, both slots). X6Y3 carries a calibrated pair on every qubit, so this runs
`relative_phase=True` around the stored value (the GE Phase's narrow-window convention).

In [ ]:
cfg = Config.from_qcal(WORK)
efp = step(EFPhase(cfg, QUBITS, CLF, points=21, span=0.25, shots=48, relative_phase=True))

for q in QUBITS:
    vz = cfg[f'qubit/{q}/EF/x90/vz']
    print(f'q{q}: EF X90 virtual-Z pair -> [{vz[0]:+.4f}, {vz[1]:+.4f}] rad')

# Two-qubit

## The drive-form seed vs the config of record

`calc_cz_frequency(form='drive')` seeds a pair's CZ tone at **(f₁₁ + f₀₂)/4** from the freshly
calibrated 1Q spectrum (spec 04 §1/§4.4) — on X6Y3's plain pairs it reproduces the calibrated freqs
to ~±100 MHz. On a **fresh bring-up** (no calibrated `CZ/freq`) you would apply it; on this
calibrated tree the config values are better, so this cell only **prints the comparison** (the
sandwich pairs activate in the shelved manifold, so their seed is expected to sit elsewhere —
spec 04 §1).

In [ ]:
cfg = Config.from_qcal(WORK)
probe = cfg.copy()
calc_cz_frequency(probe, PAIRS, state='02', form='drive')
for pair in PAIRS:
    pk = pair_key(pair)
    have = cfg[f'two_qubit/{pk}/CZ/freq'] / GHz
    seed = probe[f'two_qubit/{pk}/CZ/freq'] / GHz
    tag = '  (shelved manifold - seed not applicable)' if pair in SANDWICH else ''
    print(f'{pk}: config {have:.4f} GHz   seed {seed:.4f} GHz   Δ {1e3 * (seed - have):+7.1f} MHz{tag}')

## JAZZ — the always-on ZZ around the ring

The BIRD-echo Ramsey (spec 04 §2): on a fixed-frequency chip there is no null knob — JAZZ
*characterizes* the residual ZZ per pair, `ZZ11 = f(control=1) − f(control=0)`. The proposal writes
`two_qubit/(i, j)/ZZ11` into the working Config; note the qcal **artefact of record does not carry
it back** (`save_qcal` writes only `CZ/freq` + `CZ/pulse` — a known adapter gap), so record the
printed values.

In [ ]:
ZZ = {}
for pair in PAIRS:
    cfg = Config.from_qcal(WORK)
    r = step(JAZZ(cfg, pair, detune=2 * MHz, points=20, shots=120))
    ZZ[pair] = r.proposal.get(f'two_qubit/{pair_key(pair)}/ZZ11', float('nan'))
for pair in PAIRS:
    print(f'{pair}: ZZ11 = {ZZ[pair] / 1e3:+8.1f} kHz')

## The CZ chain — plain pairs

Per pair, the spec-04 order: `CZSweep('freq')` (the |11⟩-population dip re-pins the resonance) →
`CZFrequency` (conditionality-R argmax + parabola refine) → `RelativePhase` (the target line's tone
phase — the two-line coherent sum |E| = 2A·cos(Δφ/2) peaks R at the calibrated phase) →
`CZAmplitude` (the R error-amplification ladder — **short ladder (1, 3)**: every pulse parameter
sits behind a depth-4 posted-link queue and the unpaced higher rungs exceed the scheduled-ahead
bound, spec 04 X4) → `LocalPhases` (the per-qubit frame corrections into the channel-matched
virtual-Z entries; the conditional π is removed from the |1⟩ branch before the midpoint — the X1
parity fix). Both drive lines are swept/written **jointly** throughout (qcal's convention).

In [ ]:
for pair in PLAIN:
    print(f'=== pair {pair} ===')
    cfg = Config.from_qcal(WORK)
    step(CZSweep(cfg, pair, 'freq', span=10 * MHz, points=21, shots=120))
    cfg = Config.from_qcal(WORK)
    step(CZFrequency(cfg, pair, span=6 * MHz, points=15, ngates=1, shots=120))
    cfg = Config.from_qcal(WORK)
    step(RelativePhase(cfg, pair, points=15, ngates=1, shots=120))
    cfg = Config.from_qcal(WORK)
    step(CZAmplitude(cfg, pair, n_gates=(1, 3), window=0.3, points=11, shots=120))
    cfg = Config.from_qcal(WORK)
    step(LocalPhases(cfg, pair, points=15, shots=120))

## The CZ chain — EF-sandwich pairs (5, 6) and (6, 7)

**Prerequisite (satisfied above):** q6's `EF/freq` and `EF/x/amp` — the sandwich brackets its two
drive tones with the string-reference pre/post-pulse `single_qubit/6/EF/X/pulse` (shelve
|1⟩→|2⟩, un-shelve after), and the classes resolve it internally (`cz_sandwich`, X4) — the chain
itself is **identical**: same classes, same knobs; the kernels fold the shelve/un-shelve segments
and the partner core idles the matching windows so the tones stay lock-step.

In [ ]:
for pair in SANDWICH:
    print(f'=== pair {pair} (EF sandwich, shelf q6) ===')
    cfg = Config.from_qcal(WORK)
    step(CZSweep(cfg, pair, 'freq', span=10 * MHz, points=21, shots=120))
    cfg = Config.from_qcal(WORK)
    step(CZFrequency(cfg, pair, span=6 * MHz, points=15, ngates=1, shots=120))
    cfg = Config.from_qcal(WORK)
    step(RelativePhase(cfg, pair, points=15, ngates=1, shots=120))
    cfg = Config.from_qcal(WORK)
    step(CZAmplitude(cfg, pair, n_gates=(1, 3), window=0.3, points=11, shots=120))
    cfg = Config.from_qcal(WORK)
    step(LocalPhases(cfg, pair, points=15, shots=120))

## Spectator phases

A pair's CZ also kicks the frame of ring neighbours that sit in its pulse list as extra
virtual-Z entries (1–2 per plain pair on X6Y3). `SpectatorPhase` runs the bystander Ramsey around
one CZ fire on **three cores** and writes the spectator's channel-matched entry of the pair's own
`CZ/pulse` (X3). Spectators are read off the pulse list itself.

In [ ]:
def spectators(pair):
    '''The ring neighbours with a virtual-Z entry in this pair's CZ pulse list.'''
    out = []
    for p in cfg[f'two_qubit/{pair_key(pair)}/CZ/pulse']:
        if not isinstance(p, str) and p.get('env') == 'virtualz':
            q = int(str(p['channel']).split('.')[0][1:])
            if q not in pair:
                out.append(q)
    return out

for pair in PAIRS:
    cfg = Config.from_qcal(WORK)
    for s in spectators(pair):
        step(SpectatorPhase(cfg, pair, spectator=s, points=15, shots=120))
        cfg = Config.from_qcal(WORK)

## Validation — the conditionality R at an amplified gate count

The spec-04 §4.8-style check: R ≈ 1 at n = 1 must SURVIVE amplification (n = 3 CZs compound any
residual frequency/amplitude/phase error into a visible R drop). The four tomography sequences per
point are the calibrated chain's own machinery (`_cz_cond_R` — the `CZFrequency`/`CZAmplitude`
measurement at a pinned knob).

In [ ]:
for pair in PAIRS:
    cfg = Config.from_qcal(WORK)
    row = []
    for n in (1, 3):
        R, _ = _cz_cond_R(cfg, drv, m, pair, 'freq', _cz_freq_word(cfg, pair, m), 0, 1, n, 120)
        row.append(f'R(n={n})={float(R[0]):.3f}')
    print(f'{pair}: ' + '  '.join(row))

## Summary — the calibrated tree

Every step ran through the full qcal round trip. Diff `WORK` against `cal-config-x6y3.yaml` to see
exactly what moved.

In [ ]:
cfg = Config.from_qcal(WORK)
for q in QUBITS:
    print(f"q{q}: f_ge={cfg[f'qubit/{q}/freq'] / GHz:.6f}  f_ef={cfg[f'qubit/{q}/EF/freq'] / GHz:.6f} GHz  "
          f"x90={cfg[f'qubit/{q}/x90/amp']:.4f}  EF x90={cfg[f'qubit/{q}/EF/x90/amp']:.4f}  "
          f"EF x={cfg[f'qubit/{q}/EF/x/amp']:.4f}")
for pair in PAIRS:
    pk = pair_key(pair)
    pl = cfg[f'two_qubit/{pk}/CZ/pulse']
    drv_ents = [p for p in pl if not isinstance(p, str) and p.get('env') != 'virtualz']
    vz_ents = {p['channel']: p['kwargs'].get('phase', 0.0)
               for p in pl if not isinstance(p, str) and p.get('env') == 'virtualz'}
    print(f"{pk}: CZ {cfg[f'two_qubit/{pk}/CZ/freq'] / GHz:.4f} GHz  "
          f"amp={drv_ents[0]['kwargs']['amp']:.4f}  rel_phase={drv_ents[1]['kwargs']['phase']:+.4f}  "
          f"vz={ {c: round(v, 4) for c, v in vz_ents.items()} }")
print(f'\ncalibrated tree: {WORK}')

## Disconnect

In [ ]:
drv.close()
print('disconnected')